In [1]:
import sys
sys.path.append("/Users/livardywufianto/Projects/Trading/vscode/technical-analysis/src")

from technical_analysis.backtest import run_backtest, evaluate_run_performance
from technical_analysis.strategy import get_default_params, get_all_strategies

# Get Test DF

In [2]:
import pandas as pd
import os

# test_dir = "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/"
# ln_volume = False
# test_filepaths = [os.path.join(
#     test_dir, x) for x in os.listdir(test_dir) if str(ln_volume) in x]

test_filepaths = [
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-15_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-14_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-13_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-12_ln=False.csv", 
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-11_ln=False.csv",
]

test_filepaths = [
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-15_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-14_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-13_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-12_ln=False.csv", 
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-11_ln=False.csv",
]



df_list = [pd.read_csv(fp) for fp in test_filepaths]
test_df = pd.concat(df_list, ignore_index=True)

# Apply Strategy

In [3]:
params_dict = get_default_params()
strategies = get_all_strategies()

In [5]:
"""
Possible Exit Reason:
- Take_Profit
- Stop_Loss
- Time_Limit_48h
- Data_Cutoff_End_of_file
- Stop_Loss_Simultaneous_Hit (TP and SL hit at the same time)
"""

eval_dict_ls = []
for name, fn in strategies.items():
    buy_df = fn(test_df, params_dict)
    backtest_df = run_backtest(
        buy_df, 
        params_dict,
    )
    eval_dict = evaluate_run_performance(backtest_df, params_dict)
    eval_dict = {"strategy": name, **eval_dict}
    eval_dict_ls.append(eval_dict)

In [8]:
sliced_columns = ["strategy", 'win_rate_adjusted', 'lose_rate_adjusted', 'timeout_rate_adjusted',
       'profit_factor', 'avg_holding_hours']
pd.DataFrame(eval_dict_ls)[sliced_columns]

,strategy,win_rate_adjusted,lose_rate_adjusted,timeout_rate_adjusted,profit_factor,avg_holding_hours
0,baseline,37.65,30.62,31.73,1.48,35.6
1,shift_v1,35.39,33.90,30.71,1.11,35.0
2,shift_v2,51.02,20.41,28.57,2.30,41.7
3,kotegawa,76.92,23.08,0.00,2.85,34.4
4,momentum_filter,25.00,70.83,4.17,0.31,39.6


In [7]:
pd.DataFrame(eval_dict_ls).columns

Index(['strategy', 'z_score_threshold', 'body_ratio_threshold',
       'profit_target', 'stop_loss_target', 'max_held_hours', 'total_trades',
       'win_rate', 'lose_rate', 'timeout_rate', 'cutoff_ratet',
       'win_rate_adjusted', 'lose_rate_adjusted', 'timeout_rate_adjusted',
       'profit_factor', 'avg_holding_hours'],
      dtype='object')

In [11]:
df = buy_df.copy()
(df["ema_20"] - df["close"]) / df["ema_20"]

0        0.009442
1        0.008351
2        0.009556
3        0.009253
4        0.010899
           ...   
65671   -0.001120
65672   -0.004546
65673   -0.002699
65674    0.003741
65675    0.003385
Length: 65676, dtype: float64